In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -q transformers accelerate sentencepiece

In [3]:
# Fine-tune DeBERTa-v3-small and RoBERTa-base as 5-class sequence classifiers.
# Set SKIP_TRAIN = True and edit DEBERTA_PATH/ROBERTA_PATH below if you already have checkpoints.
import os, gc, math, numpy as np, pandas as pd, torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding, set_seed,
)
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

SEED = 42
set_seed(SEED)

SKIP_TRAIN = False   # set to True if DEBERTA_PATH and ROBERTA_PATH already exist
DEBERTA_PATH = "/kaggle/working/deberta-v3-small-ft"
ROBERTA_PATH = "/kaggle/working/roberta-base-ft"

DEBERTA_BASE = "microsoft/deberta-v3-small"
ROBERTA_BASE = "roberta-base"

MAX_LEN = 512
NUM_EPOCHS = 3
TRAIN_BS = 8
EVAL_BS = 16
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.1

ID2LETTER = ['A', 'B', 'C', 'D', 'E']
LETTER2ID = {L: i for i, L in enumerate(ID2LETTER)}

def format_input(prompt, opts):
    return (f"{prompt} A: {opts['A']} B: {opts['B']} C: {opts['C']} "
            f"D: {opts['D']} E: {opts['E']}")

def row_options(row):
    return {L: str(row[L]) for L in ID2LETTER}

train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

train_df['label'] = train_df['answer'].map(LETTER2ID)
train_df['text'] = train_df.apply(lambda r: format_input(str(r['prompt']), row_options(r)), axis=1)

# 90/10 split for internal eval during fine-tune
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(train_df))
n_val = max(1, len(train_df) // 10)
val_idx  = idx[:n_val]
trn_idx  = idx[n_val:]

trn_slice = train_df.iloc[trn_idx].reset_index(drop=True)
val_slice = train_df.iloc[val_idx].reset_index(drop=True)
print(f"train fine-tune split: train={len(trn_slice)}, val={len(val_slice)}")

def build_datasets(tokenizer):
    def tok(batch):
        return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)
    trn_hf = Dataset.from_pandas(trn_slice[['text', 'label']]).rename_column('label', 'labels')
    val_hf = Dataset.from_pandas(val_slice[['text', 'label']]).rename_column('label', 'labels')
    trn_hf = trn_hf.map(tok, batched=True, remove_columns=['text'])
    val_hf = val_hf.map(tok, batched=True, remove_columns=['text'])
    return trn_hf, val_hf

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()
    top3 = np.argsort(-logits, axis=-1)[:, :3]
    reciprocal = np.zeros(len(labels), dtype=np.float64)
    for i, lab in enumerate(labels):
        hits = np.where(top3[i] == lab)[0]
        if len(hits):
            reciprocal[i] = 1.0 / (hits[0] + 1)
    return {"accuracy": float(acc), "map@3": float(reciprocal.mean())}

def finetune_and_save(base_name, out_path):
    print(f"\n=== fine-tuning {base_name} -> {out_path} ===")
    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(base_name)
    model = AutoModelForSequenceClassification.from_pretrained(base_name, num_labels=5, torch_dtype=torch.float32)
    trn_hf, val_hf = build_datasets(tokenizer)
    total_steps = math.ceil(len(trn_hf) / TRAIN_BS) * NUM_EPOCHS
    warmup_steps = int(WARMUP_FRACTION * total_steps)
    args = TrainingArguments(
        output_dir=f"{out_path}-runs",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BS,
        per_device_eval_batch_size=EVAL_BS,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        seed=SEED,
        fp16=False,   # DeBERTa-v3 is flaky under fp16 AMP; use fp32
        dataloader_num_workers=2,
        remove_unused_columns=True,
    )
    collator = DataCollatorWithPadding(tokenizer)
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=trn_hf,
        eval_dataset=val_hf,
        processing_class=tokenizer,     # transformers 5.x: was `tokenizer=` in 4.x
        data_collator=collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    metrics = trainer.evaluate()
    print(f"final eval: {metrics}")
    trainer.save_model(out_path)
    tokenizer.save_pretrained(out_path)
    del trainer, model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if not SKIP_TRAIN:
    finetune_and_save(DEBERTA_BASE, DEBERTA_PATH)
    finetune_and_save(ROBERTA_BASE, ROBERTA_PATH)
else:
    print("SKIP_TRAIN=True -> assuming checkpoints already at DEBERTA_PATH and ROBERTA_PATH")

train fine-tune split: train=1800, val=200

=== fine-tuning microsoft/deberta-v3-small -> /kaggle/working/deberta-v3-small-ft ===


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Map@3
1,2.971502,1.897499,0.680000,0.784167
2,0.267196,0.084118,0.980000,0.990000
3,0.011730,0.005761,1.000000,1.000000


final eval: {'eval_loss': 0.005761480424553156, 'eval_accuracy': 1.0, 'eval_map@3': 1.0, 'eval_runtime': 3.3058, 'eval_samples_per_second': 60.499, 'eval_steps_per_second': 2.117, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
# Load the fine-tuned checkpoints and pre-compute the prediction tables used by Q1-Q10.
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MAX_LEN = 512
BATCH_SIZE = 16
W_DEB = 0.70
W_ROB = 0.30
INSTR = "Answer the following multiple-choice question carefully:"

ID2LETTER = ['A', 'B', 'C', 'D', 'E']
LETTER2ID = {L: i for i, L in enumerate(ID2LETTER)}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

deb_tok = AutoTokenizer.from_pretrained(DEBERTA_PATH)
deb_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH).to(device).eval()

rob_tok = AutoTokenizer.from_pretrained(ROBERTA_PATH)
rob_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH).to(device).eval()

assert deb_model.config.num_labels == 5, f"expected 5 labels, got {deb_model.config.num_labels}"
assert rob_model.config.num_labels == 5, f"expected 5 labels, got {rob_model.config.num_labels}"

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

def format_input(prompt, opts):
    return (f"{prompt} A: {opts['A']} B: {opts['B']} C: {opts['C']} "
            f"D: {opts['D']} E: {opts['E']}")

def row_options(row):
    return {L: str(row[L]) for L in ID2LETTER}

def build_texts(df, augment_prompt=False):
    texts = []
    for _, row in df.iterrows():
        p = str(row['prompt'])
        if augment_prompt:
            p = f"{INSTR} {p}"
        texts.append(format_input(p, row_options(row)))
    return texts

@torch.no_grad()
def predict_probs_batch(model, tokenizer, texts, max_len=MAX_LEN, batch_size=BATCH_SIZE):
    probs_list = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt").to(device)
        logits = model(**enc).logits
        probs_list.append(F.softmax(logits, dim=-1).cpu().numpy())
    return np.concatenate(probs_list, axis=0)

def top3_letters(probs_row):
    return [ID2LETTER[i] for i in np.argsort(-probs_row)[:3]]

def top3_string(probs_row):
    return " ".join(top3_letters(probs_row))

print("computing DeBERTa probs on all test rows...")
test_texts = build_texts(test, augment_prompt=False)
deb_test_probs = predict_probs_batch(deb_model, deb_tok, test_texts)

print("computing RoBERTa probs on all test rows...")
rob_test_probs = predict_probs_batch(rob_model, rob_tok, test_texts)

print("computing DeBERTa probs (instruction-augmented) on first 50 test rows...")
test_texts_aug50 = build_texts(test.iloc[:50], augment_prompt=True)
deb_test_probs_aug50 = predict_probs_batch(deb_model, deb_tok, test_texts_aug50)

print("computing DeBERTa+RoBERTa probs on first 100 train rows (for MAP@3)...")
train_texts_100 = build_texts(train.iloc[:100], augment_prompt=False)
deb_train100_probs = predict_probs_batch(deb_model, deb_tok, train_texts_100)
rob_train100_probs = predict_probs_batch(rob_model, rob_tok, train_texts_100)

print(f"shapes -> test deb: {deb_test_probs.shape}, test rob: {rob_test_probs.shape}, "
      f"deb aug50: {deb_test_probs_aug50.shape}, train100 deb: {deb_train100_probs.shape}")

device: cuda


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

computing DeBERTa probs on all test rows...
computing RoBERTa probs on all test rows...
computing DeBERTa probs (instruction-augmented) on first 50 test rows...
computing DeBERTa+RoBERTa probs on first 100 train rows (for MAP@3)...
shapes -> test deb: (500, 5), test rob: (500, 5), deb aug50: (50, 5), train100 deb: (100, 5)


In [5]:
# Q1: DeBERTa top-1 for test row 25
p_deb_25 = deb_test_probs[25]
top1_deb_25 = int(np.argmax(p_deb_25))
print(f"Q1: {ID2LETTER[top1_deb_25]}, {p_deb_25[top1_deb_25]:.4f}")
print(f"    full probs (A,B,C,D,E): {p_deb_25}")

Q1: E, 0.5902
    full probs (A,B,C,D,E): [0.05997877 0.03500944 0.2771886  0.03764074 0.5901825 ]


In [6]:
# Q2: simple average of DeBERTa + RoBERTa probs, top-1 for test row 25
p_rob_25 = rob_test_probs[25]
p_avg_25 = (p_deb_25 + p_rob_25) / 2
top1_avg_25 = int(np.argmax(p_avg_25))
print(f"Q2: {ID2LETTER[top1_avg_25]}")
print(f"    avg probs: {p_avg_25}")

Q2: E
    avg probs: [0.03044395 0.01787488 0.13891925 0.0191799  0.7935821 ]


In [7]:
# Q3: weighted 0.70/0.30 ensemble top-1 for test row 25
p_wens_25 = W_DEB * p_deb_25 + W_ROB * p_rob_25
top1_wens_25 = int(np.argmax(p_wens_25))
print(f"Q3: {ID2LETTER[top1_wens_25]}")
print(f"    weighted probs: {p_wens_25}")

Q3: E
    weighted probs: [0.04225788 0.02472871 0.19422698 0.02656423 0.7122222 ]


In [8]:
# Q4: Top-3 prediction string for test row 25 under weighted ensemble
q4 = top3_string(p_wens_25)
print(f"Q4: {q4}")

Q4: E C A


In [9]:
# Q5: full submission using weighted ensemble on all 500 test rows
wens_test_probs = W_DEB * deb_test_probs + W_ROB * rob_test_probs
preds = [top3_string(wens_test_probs[i]) for i in range(len(test))]

# match sample_submission column names (ID, Prediction)
sub = pd.DataFrame({"ID": test["id"], "Prediction": preds})
sub.to_csv("submission.csv", index=False)

q5 = len(sub)
print(f"Q5: {q5}")
print(sub.head())

Q5: 500
   ID Prediction
0   1      A D E
1   2      B D C
2   3      B D C
3   4      E C A
4   5      C E B


In [10]:
# Q6: TTA on DeBERTa for first 50 test rows.
# Top-1 changed = argmax(orig probs) vs argmax(mean(orig, aug) probs)
deb_test_probs_50 = deb_test_probs[:50]
deb_tta_avg_50 = (deb_test_probs_50 + deb_test_probs_aug50) / 2

orig_top1 = np.argmax(deb_test_probs_50, axis=1)
tta_top1  = np.argmax(deb_tta_avg_50, axis=1)
q6 = int((orig_top1 != tta_top1).sum())
print(f"Q6: {q6}")

Q6: 4


In [11]:
# Q7: DeBERTa vs weighted-ensemble top-1 disagreement on first 100 test rows
deb100  = deb_test_probs[:100]
wens100 = wens_test_probs[:100]
deb_top1_100  = np.argmax(deb100, axis=1)
wens_top1_100 = np.argmax(wens100, axis=1)
q7 = int((deb_top1_100 != wens_top1_100).sum())
print(f"Q7: {q7}")

Q7: 6


In [12]:
# Q8: rows with positive confidence gain on first 100 test rows
conf_deb  = deb100.max(axis=1)
conf_wens = wens100.max(axis=1)
q8 = int(((conf_wens - conf_deb) > 0).sum())
print(f"Q8: {q8}")

Q8: 97


In [13]:
# Q9: rows with ANY change in ordered Top-3 (DeBERTa vs weighted ensemble), first 100 test rows
q9 = 0
for i in range(100):
    if top3_string(deb100[i]) != top3_string(wens100[i]):
        q9 += 1
print(f"Q9: {q9}")

Q9: 7


In [14]:
# Q10: MAP@3 on first 100 train rows using weighted ensemble.
# (test.csv has no labels; MAP@3 requires labels -> use train[:100].)
def map3_row(pred_letters, correct):
    return 1.0 / (pred_letters.index(correct) + 1) if correct in pred_letters else 0.0

wens_train100_probs = W_DEB * deb_train100_probs + W_ROB * rob_train100_probs
scores = []
for i in range(100):
    top3 = top3_letters(wens_train100_probs[i])
    correct = str(train.iloc[i]['answer'])
    scores.append(map3_row(top3, correct))

q10 = round(float(np.mean(scores)), 4)
print(f"Q10: {q10}")

Q10: 0.995
